This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace. 

In [14]:
!pip3 install -r requirements.txt

  Using cached langchain-0.0.305-py3-none-any.whl (1.8 MB)
  Using cached pytest-8.3.2-py3-none-any.whl (341 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
  Using cached jupyter-1.0.0-py2.py3-none-any.whl (2.7 kB)
     |████████████████████████████████| 80 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 4.1 MB/s eta 0:00:01
     |████████████████████████████████| 56 kB 15.3 MB/s eta 0:00:01
  Using cached numexpr-2.10.1-cp39-cp39-macosx_11_0_arm64.whl (130 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached aiohttp-3.10.5-cp39-cp39-macosx_11_0_arm64.whl (389 kB)
  Using cached ipykernel-6.29.5-py3-none-any.whl (117 kB)
     |████████████████████████████████| 123 kB 76.6 MB/s eta 0:00:01
  Using cached jupyter_console-6.6.3-py3-no

In [1]:
from langchain_community.llms import OpenAI
from langchain.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain.chains.conversational_retrieval.base import ConversationalRetrievalChain
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field, NonNegativeInt
from comet_ml import Experiment
from fastapi.encoders import jsonable_encoder
import torch
from diffusers import StableDiffusionPipeline
from langchain.output_parsers import PydanticOutputParser
import PIL
import time
from tqdm import tqdm
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import google.generativeai as genai
import os
from dotenv import load_dotenv
import pandas as pd
from typing import List
import requests

# Load environment variables
load_dotenv('my_config.env')

# API configuration
API_KEY = os.getenv('API_KEY')
openai_api_key = os.getenv("OPENAI_API_KEY")
COMET_API_KEY = os.getenv("COMET_API_KEY")


# Initialize the experiment
experiment = Experiment(
    api_key=COMET_API_KEY,
    project_name="real-estate-agent",
    workspace="polarbeargo",
    log_code=True,
)

/Users/hsin-wenchang/Documents/GitHub/GenAIND-Project-Personalized-Real-Estate-Agent/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/polarbeargo/real-estate-agent/2091f4b093554ad4ae6223fbc0515a40



- Define the prompt for generating synthetic real estate data (images and text)

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [2]:
instruction = """
Generate eighteen realistic real estate listings from a wide range of neighborhoods.
"""

template = \
"""
Here is the template of real estate listing:

Neighborhood: Mountain View
Price: $650,000
Bedrooms: 5
Bathrooms: 4
House Size: 3000 sqft
Description: Spacious family home with breathtaking views of the mountains and a large backyard for outdoor entertaining.
Neighborhood Description: Mountain View is known for its scenic landscape and outdoor activities, making it the ideal location for nature lovers and adventure seekers.
"""
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7, api_key=openai_api_key, max_tokens = 500)
image_model = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")

# For Apple Silicon (M1/M2) replace mps to device if executing on other devices
image_model.to("mps")

image_dir = "generated_images"
os.makedirs(image_dir, exist_ok=True)

# Define the Listing data model
class Listing(BaseModel):
    neighborhood: str = Field(description="The neighborhood where the property is located.")
    price: NonNegativeInt = Field(description="The price of the property in USD.")
    bedrooms: NonNegativeInt = Field(description="The number of bedrooms in the property.")
    bathrooms: NonNegativeInt = Field(description="The number of bathrooms in the property.")
    house_size: NonNegativeInt = Field(description="The size of the property in square feet.")
    description: str = Field(description="A brief description of the property.")
    neighborhood_description: str = Field(description="A description of the neighborhood where the property is located.")

class Listings(BaseModel):
    listing: List[Listing] = Field(description="List of available real estate listings.")
    

def create_listing_prompt(listing: Listing) -> str:
    return f"""
    Neighborhood: {listing.neighborhood}
    Price: ${listing.price}
    Bedrooms: {listing.bedrooms}
    Bathrooms: {listing.bathrooms}
    House Size: {listing.house_size} sqft
    Description: {listing.description}
    Neighborhood Description: {listing.neighborhood_description}
    """

# Define few-shot examples
examples = [
    {
        "question": "Generate a listing for a 3-bedroom house in downtown.",
        "answer": Listing(
            neighborhood="Downtown",
            price=500000,
            bedrooms=3,
            bathrooms=2,
            house_size=1500,
            description="A beautiful 3-bedroom house located in the heart of downtown with modern amenities.",
            neighborhood_description="Downtown is vibrant and bustling, with plenty of restaurants, shops, and parks."
        )
    },
    {
        "question": "Create a listing for a luxury apartment in the suburbs.",
        "answer": Listing(
            neighborhood="Suburbia",
            price=750000,
            bedrooms=2,
            bathrooms=2,
            house_size=1200,
            description="A luxurious apartment featuring high-end finishes and spacious living areas.",
            neighborhood_description="Suburbia offers a peaceful environment with great schools and family-friendly parks."
        )
    }
]

parser = PydanticOutputParser(pydantic_object=Listings)

example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="{question}\n{answer}",
    partial_variables={"format_instructions": parser.get_format_instructions},
)

few_shot_prompt = FewShotPromptTemplate(
    examples=[{"question": ex["question"], "answer": create_listing_prompt(ex["answer"])} for ex in examples],
    example_prompt=example_prompt,
    suffix="Use these examples to generate a listing for the following question: {input}",
    input_variables=["input"],
    partial_variables={"format_instructions": parser.get_format_instructions},
)

full_prompt = few_shot_prompt.format(sample=template, input=instruction)
response = llm(full_prompt)
print(f"Raw Response: {response}")  # Debugging line to check the response format

/var/folders/f8/sxbz4hwx6js37sqlrvxpmq1m0000gn/T/ipykernel_19504/866001111.py:17: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7, api_key=openai_api_key, max_tokens = 500)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/Users/hsin-wenchang/Documents/GitHub/GenAIND-Project-Personalized-Real-Estate-Agent/env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/var/folders/f8/sxbz4hwx6js37sqlrvxpmq1m0000gn/T/ipykernel_19504/866001111.py:96: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(full_prompt)


Raw Response: 
1. Neighborhood: Midtown
   Price: $1,200,000
   Bedrooms: 4
   Bathrooms: 3
   House Size: 2500 sqft
   Description: Stunning 4-bedroom home in the highly desirable Midtown neighborhood, featuring a chef's kitchen and rooftop terrace with city views.
   Neighborhood Description: Midtown is a bustling and trendy area with a lively nightlife, upscale restaurants, and convenient access to public transportation.

2. Neighborhood: Historic District
   Price: $850,000
   Bedrooms: 3
   Bathrooms: 2
   House Size: 1800 sqft
   Description: Charming 3-bedroom home in the historic district, with original hardwood floors and a private backyard oasis.
   Neighborhood Description: The Historic District is known for its beautiful architecture, rich history, and quaint shops and cafes.

3. Neighborhood: Beachfront
   Price: $2,500,000
   Bedrooms: 5
   Bathrooms: 4
   House Size: 4000 sqft
   Description: Luxurious 5-bedroom beachfront property with panoramic ocean views, a private p

In [3]:
# Split the string into individual listings
listings = response.strip().split('\n\n')
print(listings)  # Debugging line to check the listings format
data = []

for listing in listings:

    # Remove the integer before 'Neighborhood'
    listing = listing.split('. ', 1)[1]

    # Split the listing into lines
    lines = listing.split('\n')
    listing_data = {}
    
    for line in lines:
        # Check if the line contains a colon
        if ': ' in line:
            # Split on the first colon
            key, value = line.split(': ', 1)  
            listing_data[key.strip()] = value.strip()
    
    data.append(listing_data)

df = pd.DataFrame(data)

df.rename(columns={
    'Neighborhood': 'neighborhood',
    'Price': 'price',
    'Bedrooms': 'bedrooms',
    'Bathrooms': 'bathrooms',
    'House Size': 'house_size',
    'Description': 'description',
    'Neighborhood Description': 'neighborhood_description'
}, inplace=True)

# Convert price to integer and house_size to integer (removing ' sqft')
df['price'] = df['price'].replace({'\$': '', ',': ''}, regex=True).astype(int)
df['house_size'] = df['house_size'].replace({' sqft': ''}, regex=True).fillna(0).astype(int)
df


["1. Neighborhood: Midtown\n   Price: $1,200,000\n   Bedrooms: 4\n   Bathrooms: 3\n   House Size: 2500 sqft\n   Description: Stunning 4-bedroom home in the highly desirable Midtown neighborhood, featuring a chef's kitchen and rooftop terrace with city views.\n   Neighborhood Description: Midtown is a bustling and trendy area with a lively nightlife, upscale restaurants, and convenient access to public transportation.", '2. Neighborhood: Historic District\n   Price: $850,000\n   Bedrooms: 3\n   Bathrooms: 2\n   House Size: 1800 sqft\n   Description: Charming 3-bedroom home in the historic district, with original hardwood floors and a private backyard oasis.\n   Neighborhood Description: The Historic District is known for its beautiful architecture, rich history, and quaint shops and cafes.', '3. Neighborhood: Beachfront\n   Price: $2,500,000\n   Bedrooms: 5\n   Bathrooms: 4\n   House Size: 4000 sqft\n   Description: Luxurious 5-bedroom beachfront property with panoramic ocean views, a p

,neighborhood,price,bedrooms,bathrooms,house_size,description,neighborhood_description
0,Midtown,1200000,4,3,2500,Stunning 4-bedroom home in the highly desirabl...,Midtown is a bustling and trendy area with a l...
1,Historic District,850000,3,2,1800,Charming 3-bedroom home in the historic distri...,The Historic District is known for its beautif...
2,Beachfront,2500000,5,4,4000,Luxurious 5-bedroom beachfront property with p...,The Beachfront neighborhood offers a luxurious...
3,Suburban Oasis,600000,4,3,3000,Spacious 4-bedroom home in a peaceful suburban...,Suburban Oasis is a family-friendly neighborho...
4,Arts District,1000000,2,2,1500,Chic 2-bedroom loft in the vibrant Arts Distri...,The Arts District is a bustling and artistic c...
5,Lakefront,1800000,6,5,6000,Spect,NaN


In [4]:
df.to_csv('generated_real_estate_data.csv', index_label = 'id')

In [6]:
# Batch generation of images based on the DataFrame
def generate_images(df, batch_size=2):
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i + batch_size]
        for idx, row in batch.iterrows():
            prompt = f"A {row['bedrooms']}-bedroom house in {row['neighborhood']}. {row['neighborhood_description']}"
            image = image_model(prompt, num_inference_steps=50).images[0]
            image_path = os.path.join(image_dir, f"{row['neighborhood']}_{row['bedrooms']}_bedroom.png")
            image.save(image_path)
            print(f"Generated image saved at: {image_path}")

generate_images(df)


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Midtown_4_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Historic District_3_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

...

Generated image saved at: generated_images/Suburban Oasis_4_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Arts District_2_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Lakefront_6_bedroom.png


##### Multimodal Vector Database Store, Embeddings Transformation and Semantic Search

In [ ]:
loader = CSVLoader(file_path='./generated_real_estate_data.csv')
docs = loader.load()

- Storing Listings Into a Vector Database

In [ ]:
idx = [i for i in range(len(df))]
image_paths = []
images = []
texts = []
text_template = """
Neighborhood: {}
Price: {}
Bedrooms: {}
Bathrooms: {}
House Size: {}

Description: {}
Neighborhood Description: {}
"""

for i, row in df.iterrows():
    image_path = os.path.join(image_dir, f"{row['neighborhood']}_{i}_listing.png")
    image_paths.append(image_path)
    images.append(PIL.Image.open(image_path))
    texts.append(text_template.format(row['neighborhood'], row['price'], row['bedrooms'], row['bathrooms'], row['house_size'], row['description'], row['neighborhood_description']))

In [ ]:
clip_embeddings = OpenAIClipEmbeddings()

def get_clip_embeddings(texts):
    return clip_embeddings.embed_documents(texts)

clip_embeddings_result = get_clip_embeddings(texts)
print(clip_embeddings_result)

chroma_client = chromadb.Client()
clip_db = chroma_client.get_or_create_collection(name="clip_embeddings")

clip_db.add(
    documents=texts,
    embeddings=clip_embeddings_result,
    ids=idx
)

clip_db.add(
    documents=image_paths,
    embeddings=clip_embeddings_result,
    ids=idx
)

In [ ]:
questions = [   
                "How big do you want your house to be?" 
                "What are 3 most important things for you in choosing this property?", 
                "Which amenities would you like?", 
                "Which transportation options are important to you?",
                "How urban do you want your neighborhood to be?",   
            ]
answers = [
    "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood, good local schools, and convenient shopping options.",
    "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
    "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

In [ ]:
query_text = "modern apartment with high-end finishes"
query_embedding = get_clip_embeddings([query_text])[0]
results = clip_db.query(query_embedding, n_results=1)
print(results)

- Integrate kubeflow pipelines

In [ ]:
%%writefile HomeMatch.py

- Use Gradio to create an interactive interface where we can input data related to home preferences, and the model can predict the best match for them.

In [ ]:
import gradio as gr 

In [ ]:

def generate_listing(neighborhood, price, bedrooms, bathrooms, house_size, description, neighborhood_description):
        return {
            "Neighborhood": neighborhood,
            "Price": price,
            "Bedrooms": bedrooms,
            "Bathrooms": bathrooms,
            "House Size": house_size,
            "Description": description,
            "Neighborhood Description": neighborhood_description
        }

def predict_best_match(neighborhood, price, bedrooms, bathrooms, house_size, description, neighborhood_description):
    listing = generate_listing(neighborhood, price, bedrooms, bathrooms, house_size, description, neighborhood_description)
    # Here you can add the logic to predict the best match using the model
    return listing

interface = gr.Interface(
    fn=predict_best_match,
    inputs=[
        gr.inputs.Textbox(label="Neighborhood"),
        gr.inputs.Number(label="Price"),
        gr.inputs.Number(label="Bedrooms"),
        gr.inputs.Number(label="Bathrooms"),
        gr.inputs.Number(label="House Size"),
        gr.inputs.Textbox(label="Description"),
        gr.inputs.Textbox(label="Neighborhood Description")
    ],
    outputs="json",
    title="Home Match Predictor",
    description="Input your home preferences to find the best match."
)

In [ ]:
if __name__ == "__main__":
    interface.launch()

In [2]:
experiment.end()

NameError: name 'experiment' is not defined